Title: check_CMIP6_models.ipynb

Purpose: Look at some CMIP6 models

Author: Onno Nennecke on 15.03.2025 Modified: 27.03.2025


In [1]:
import esmvalcore.esgf
import numpy as np
import pandas as pd
import xarray as xr
import glob
import os
from collections import Counter

In [2]:
allESMs = ['ACCESS-CM2', 'ACCESS-ESM1-5', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CanESM5', 'CESM2', 'CESM2-WACCM', 'CNRM-CM6-1', 'CNRM-CM6-1-HR',
           'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-Veg', 'FGOALS-f3-L', 'FGOALS-g3', 'GFDL-CM4', 'GFDL-ESM4',
        'GISS-E2-1-G', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'INM-CM4-8', 'INM-CM5-0', 'IPSL-CM6A-LR', 'KACE-1-0-G', 'MIROC6',
        'MIROC-ES2L', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NESM3', 'NorESM2-LM', 'TaiESM1', 'UKESM1-0-LL']
selESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'CESM2-WACCM', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-Veg',
           'GFDL-CM4', 'GFDL-ESM4', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'KACE-1-0-G', 'TaiESM1', 'UKESM1-0-LL']
# selESMs = ['EC-Earth3-Veg']

no_run_ESMs = ['CAMS-CSM1-0', 'CESM2-WACCM', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'FGOALS-f3-L', 'GFDL-CM4', 'GISS-E2-1-G', 'HadGEM3-GC31-LL',
               'HadGEM3-GC31-MM', 'NESM3']

no_nothing_ESMs = ['FGOALS-f3-L', 'GFDL-CM4', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'NESM3']



In [3]:
downloaded_files = []
for ESM in selESMs:
    fls = esmvalcore.esgf.find_files(
        project='CMIP6',
        mip='day',
        short_name='tas',
        dataset=ESM,
        exp='ssp370',
        ensemble='*',
    )  
    ensemble_names = np.unique([fl.__dict__['facets']['ensemble'] for fl in fls])
    print('1. print', ESM, ':', ensemble_names)

    
    print(ESM, ':\n')

    for ensemble in ensemble_names:
        print('ensemble:', ensemble)
        download = True
        files_to_download = []
        for var in ['tas','rsds','sfcWind', 'tasmax', 'psl']: # , 'zg'
            # print(var)
            fls = esmvalcore.esgf.find_files(
                project='CMIP6',
                mip='day',
                short_name=var,
                dataset=ESM,
                exp='ssp370',
                ensemble=ensemble,
            )
            print('2. print', ESM, ':', ensemble, ':', var, ':', len(fls))
            useful_files_for_variable = []
            for fl in fls:
                useful_files_for_variable.append(fl)
                if int(fl.name.split('-')[-1][:4]) >= 2024:
                    break
            files_to_download += useful_files_for_variable

            if len(useful_files_for_variable) == 0:
                download = False
                print('No files found for', var)
                break


        if download:
            # print(ESM,files_to_download)
            print('Download', ensemble)
            for fl in files_to_download:
                print(fl)
            downloaded_files.append(files_to_download)
            # esmvalcore.esgf.download(files_to_download, '/climca/data/', n_jobs=4)

print(len(downloaded_files))



1. print ACCESS-CM2 : ['r10i1p1f1' 'r1i1p1f1' 'r2i1p1f1' 'r3i1p1f1' 'r4i1p1f1' 'r5i1p1f1'
 'r6i1p1f1' 'r7i1p1f1' 'r8i1p1f1' 'r9i1p1f1']
ACCESS-CM2 :

ensemble: r10i1p1f1
2. print ACCESS-CM2 : r10i1p1f1 : tas : 2
2. print ACCESS-CM2 : r10i1p1f1 : rsds : 0
No files found for rsds
ensemble: r1i1p1f1
2. print ACCESS-CM2 : r1i1p1f1 : tas : 2
2. print ACCESS-CM2 : r1i1p1f1 : rsds : 2
2. print ACCESS-CM2 : r1i1p1f1 : sfcWind : 2
2. print ACCESS-CM2 : r1i1p1f1 : tasmax : 2
2. print ACCESS-CM2 : r1i1p1f1 : psl : 2
Download r1i1p1f1
ESGFFile:CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r1i1p1f1/day/tas/gn/v20191108/tas_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20150101-20641231.nc on hosts ['esgf-data1.llnl.gov', 'esgf-data04.diasjp.net', 'esgf.ceda.ac.uk', 'esgf.nci.org.au', 'esgf3.dkrz.de']
ESGFFile:CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r1i1p1f1/day/rsds/gn/v20191108/rsds_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20150101-20641231.nc on hosts ['esgf-data04.diasjp.net', 'esgf-data1.llnl.gov', 'e

In [ ]:
ESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'EC-Earth3', 'GFDL-ESM4', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'KACE-1-0-G', 'TaiESM1', 'UKESM1-0-LL'] # 'EC-Earth3-Veg'
ESMs = ['UKESM1-0-LL']

# Define base path pattern
base_path = '/climca/data/CMIP6/ScenarioMIP/*/{}/ssp370/*/day/*/*/*/'

# Loop through ESMs and collect counts for each ESM
for ESM in ESMs:
    print(f"\n=== ESM: {ESM} ===")

    # List to store folder names for the current ESM
    folder_names = []
    
    # Find matching paths
    paths = glob.glob(base_path.format(ESM))
    for path in paths:
        print('paths:', path)
        folder = os.path.basename(os.path.normpath(path))  # Get the folder name
        folder_names.append(folder)

    # Count unique folder names
    folder_count = Counter(folder_names)

    # Print results for this ESM
    if folder_count:
        # print("Unique folder names and their counts:")
        for folder, count in folder_count.items():
            print(f"  {folder}: {count}")
    else:
        print("  No matching folders found.")



=== ESM: UKESM1-0-LL ===
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r4i1p1f2/day/sfcWind/gn/v20190813/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r4i1p1f2/day/tasmax/gn/v20190813/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r4i1p1f2/day/rsds/gn/v20190813/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r4i1p1f2/day/tas/gn/v20190813/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r4i1p1f2/day/psl/gn/v20190813/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r19i1p1f2/day/sfcWind/gn/v20200608/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r19i1p1f2/day/tasmax/gn/v20200608/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r19i1p1f2/day/rsds/gn/v20200608/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r19i1p1f2/day/tas/gn/v20200608/
paths: /climca/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r19i1p1f2/day/psl/gn/v20200608/
paths: /climca/data/CMI

----

#### Check CMIP6 data (Resolution and units)

In [3]:
# Read the dataframe from the csv file
df = pd.read_csv('/home/onennecke/CMIP_models/CMIP6_runs.csv')
df['Ref'] = df.groupby(['ESM', 'Institution']).cumcount().apply(lambda x: 1 if x == 0 else 0)

# Load climate data

MIP = 'ScenarioMIP' # CMIP

# Institution = df['Institution']
# # Institution = ['CSIRO-ARCCSS', 'BCC', 'NCAR', 'EC-Earth-Consortium', 'NOAA-GFDL', 'NIMS-KMA', 'DKRZ', 'MRI', 'AS-RCEC', 'MOHC', 'NIMS-KMA']
# # Institution = ['CSIRO-ARCCSS']

# ESMs = df['ESM']
# # ESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'EC-Earth3', 'GFDL-ESM4', 'KACE-1-0-G', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'TaiESM1', 'UKESM1-0-LL', 'UKESM1-0-LL']
# # ESMs = ['ACCESS-CM2']

# run = df['run']
# # run = 'r1i1p1f1'

scenario = 'ssp370'
time_res = 'day'
variables = ['sfcWind', 'rsds', 'tas', 'tasmax'] # List of variables
# variables = ['sfcWind', 'rsds', 'tas', 'tasmax', 'psl'] # List of variables
variables = ['rsds', 'tas', 'tasmax'] # List of variables

grid_def = '*'
version = '*'

In [4]:
for i in range(len(df)):
    if df['Ref'][i] == 1:
        ESM = df['ESM'][i]
        Inst = df['Institution'][i]
        run = df['run'][i]
        print(f'Processing Run Nr. {i+1}, {ESM}, {Inst}, {run}, \n')
        
        
        for var in variables:
            print(f'Processing variable: {var}')
            path = f'/climca/data/CMIP6/{MIP}/{Inst}/{ESM}/{scenario}/{run}/{time_res}/{var}/{grid_def}/{version}/{var}_{time_res}_{ESM}_{scenario}_{run}_*'
            files = [f for f in glob.glob(path) if f.endswith('.nc')]

            nc = xr.open_mfdataset(files)
            
            var_tst = nc
            # print(var_tst.lat.values)
            # print(var_tst.lon.values)

            print((var_tst.lat.values[1:]- var_tst.lat.values[:-1])[0:10])
            print((var_tst.lon.values[1:]- var_tst.lon.values[:-1])[0:10])

            # print((var_tst.time.values[1:10] - var_tst.time.values[0:9]) / np.timedelta64(1, 'h'))
            
            break
    break

Processing Run Nr. 1, ACCESS-CM2, CSIRO-ARCCSS, r4i1p1f1, 

Processing variable: rsds
[1.25 1.25 1.25 1.25 1.25 1.25 1.25 1.25 1.25 1.25]
[1.875 1.875 1.875 1.875 1.875 1.875 1.875 1.875 1.875 1.875]


In [13]:
nc
# print(nc['rsds'].attrs['variable_id'])       # should be "rsds"
print(nc['rsds'].attrs.get('standard_name')) # often "surface_downwelling_shortwave_flux_in_air"
print(nc['rsds'].attrs['long_name'])         # e.g. "Surface downwelling shortwave radiation"
print(nc['rsds'].attrs['units'])             # should be "W m-2"
nc['rsds']


surface_downwelling_shortwave_flux_in_air
Surface Downwelling Shortwave Radiation
W m-2


<xarray.DataArray 'rsds' (time: 18263, lat: 144, lon: 192)> Size: 2GB
dask.array<open_dataset-rsds, shape=(18263, 144, 192), dtype=float32, chunksize=(1, 144, 192), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 146kB 2015-01-01T12:00:00 ... 2064-12-31T1...
  * lat      (lat) float64 1kB -89.38 -88.12 -86.88 -85.62 ... 86.88 88.12 89.38
  * lon      (lon) float64 2kB 0.9375 2.812 4.688 6.562 ... 355.3 357.2 359.1
Attributes:
    standard_name:  surface_downwelling_shortwave_flux_in_air
    long_name:      Surface Downwelling Shortwave Radiation
    comment:        Surface solar irradiance for UV calculations.
    units:          W m-2
    cell_methods:   area: time: mean
    cell_measures:  area: areacella

In [6]:
var_tst = nc
# print(var_tst.lat.values)
# print(var_tst.lon.values)

print((var_tst.lat.values[1:]- var_tst.lat.values[:-1])[0:10])
print((var_tst.lon.values[1:]- var_tst.lon.values[:-1])[0:10])

print((var_tst.time.values[1:10] - var_tst.time.values[0:9]) / np.timedelta64(1, 'h'))

[1.25 1.25 1.25 1.25 1.25 1.25 1.25 1.25 1.25 1.25]
[1.875 1.875 1.875 1.875 1.875 1.875 1.875 1.875 1.875 1.875]
[24. 24. 24. 24. 24. 24. 24. 24. 24.]
